# Quickstart

A guided tour of this corpus for someone who has just cloned the repository. By the end you will
have loaded the dataset, listed what it contains, read a verse in four scripts, compared the same
verse across six manuscripts, run a search, and filtered out the material the test suite does not
validate.

## What the corpus is

Five Targumim, the Aramaic translations of the Pentateuch, published as one dataset of aligned
manuscript transcriptions with a morphological analysis under every word:

| witness | what it is |
|---|---|
| Targum Onqelos | the standard Babylonian Targum |
| Targum Pseudo-Jonathan | an expansive Palestinian Targum |
| Targum Neofiti | a complete Palestinian Targum, published in two views |
| the Fragment Targums | what survives of a further Palestinian tradition |
| the Cairo Genizah fragments | 39 fragments, the oldest witnesses here |

Neofiti appears twice because the manuscript carries marginal and interlinear additions. The
`base_raw` view keeps only unmarked text, and the `full_text` view places every reading side by
side. They are two readings of one codex, not two manuscripts.

The text is Jewish Aramaic, Babylonian for Onqelos and Palestinian for the rest, carried both in
Aramaic square script and in ETCBC transliteration. Aramaic is agglutinative, so what looks like
one word is often several morphemes, and the dataset's distinctive feature is that it decomposes
them: a surface word is a `word_group`, and the morphemes under it are `word` nodes, each with its
own lexeme, part of speech and gloss.

## What you need

* **Python 3.14** and the pinned dependencies. The project uses `pipenv`, so from the repository
  root: `pipenv install` then `pipenv shell`. The two libraries that matter here are
  `context-fabric` for reading and `text-fabric` for compiling.
* **The data, which is already in the clone.** `tf/0.1` holds 48 feature files tracked directly
  in git, so a normal `git clone` brings the corpus with it. Nothing needs downloading and no
  database is required to read it. If `tf/0.1` arrives empty, run `git lfs install` and then
  `git lfs pull`, or clone again.
* **A database only if you intend to rebuild.** PostgreSQL is the source of truth and `tf/` is
  generated from it by `src/main/pipeline/tf_pipeline.py` and `programs/xml_to_tf.py`. Reading, which
  is all this notebook does, never touches it.

Run the cells in order. The first one is the only one that takes a moment.

In [ ]:
# Load the corpus.
#
# The dataset is addressed by path, not by an organisation and repository name, so this works in a
# fresh clone with no environment file and no network access. `locations` is the directory holding
# the versioned datasets and `modules` is the version inside it.
#
# The version is read from the application configuration rather than written here as a literal, so
# this notebook cannot end up describing an older build than the repository ships.
#
# `silent="deep"` suppresses one log line per feature. Drop it if you want to watch the load.

from pathlib import Path

import yaml
import cfabric

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config/xml.yml").exists())
TF_VERSION = yaml.safe_load((REPO_ROOT / "app/config.yaml").read_text())["provenanceSpec"]["version"]

CF = cfabric.Fabric(locations=str(REPO_ROOT / "tf"), modules=TF_VERSION, silent="deep")
api = CF.loadAll(silent="deep")

# The five handles you will use for everything.
#   F  features of a node          F.lex.v(word)
#   E  edges between nodes         E.crossref.f(verse)
#   L  locality, up and down       L.d(verse, otype="word"), L.u(word, otype="verse")
#   T  text and sections           T.text(nodes), T.nodeFromSection(("Genesis", 1, 1))
#   S  search                      S.search(template)
F, E, L, T, S = api.F, api.E, api.L, api.T, api.S
Fs = api.Fs                        # Fs("lex") when the feature name is a variable

print(f"loaded version {TF_VERSION} from {REPO_ROOT / 'tf' / TF_VERSION}")
print(f"{len(F.otype.all)} node types, {len(set(api.Fall()) | set(api.Eall()))} features")

## What is in it

The dataset is a hierarchy of node types. Each node type is a tier of the edition, and every tier
is addressable on its own.

In [ ]:
# The node types, outermost first, with what each one is.

WHAT = {
    "text": "one manuscript under one reading view",
    "book": "a book of the Pentateuch within one manuscript",
    "chapter": "a chapter within one book",
    "verse": "one line of transcription, addressed by book, chapter and verse",
    "segment": "a run of the line sharing one editorial state",
    "word_group": "one surface word as the manuscript writes it",
    "word": "one morpheme, the unit the morphology is attached to",
    "lexeme": "a dictionary headword, shared across the whole corpus",
}
for otype in F.otype.all:
    print(f"  {otype:<11} {len(list(F.otype.s(otype))):>9,}   {WHAT.get(otype, '')}")

print()
print("the words are the slots, so every other tier is a span of words:")
_v = T.nodeFromSection(("Genesis", 1, 1))
print(f"  Genesis 1:1 covers {len(L.d(_v, otype='word'))} words "
      f"in {len(L.d(_v, otype='word_group'))} surface words "
      f"and {len(L.d(_v, otype='segment'))} segments")

In [ ]:
# Which features each tier carries. This is the map to consult when you want a value and do not
# know where it lives. Sampled over the first 400 nodes of each type, so a feature that only a
# rare node carries may not show.

FEATURES = sorted(f for f in (set(api.Fall()) | set(api.Eall()))
                  if f not in ("otype", "oslots") and f != "otext")
for otype in F.otype.all:
    sample = F.otype.s(otype)[:400]
    present = [f for f in FEATURES
               if hasattr(F, f) and any(Fs(f).v(n) is not None for n in sample)]
    print(f"{otype:<11} {', '.join(present)}")

print()
print("the four text formats, on the opening words of the corpus:")
for fmt in sorted(T.formats):
    print(f"  {fmt:<16} {T.text(range(1, 9), fmt=fmt)}")

## Reading a verse

`T.nodeFromSection` resolves a section address, and there is one thing to know about it before you
trust the answer. The section tier is `book, chapter, verse` and it does **not** include the
manuscript, so an address resolves to whichever witness sorts first, which is Onqelos. That is
deliberate, but it means section lookup is not how you reach a particular manuscript.

In [ ]:
# One address, read the two ways.

_node = T.nodeFromSection(("Genesis", 44, 18))
_witness = F.text.v(L.u(_node, otype="text")[0])
print(f"T.nodeFromSection(('Genesis', 44, 18)) -> node {_node}, which is {_witness}")
print()
for fmt in ("text-orig-full", "text-trans-full", "text-cons", "text-trans-cons"):
    print(f"  {fmt:<16} {T.text(_node, fmt=fmt)[:70]}")

print()
print("to reach a named manuscript, walk the text tier instead:")


def verses_of(text_name):
    """Every verse node of one manuscript, in publication order."""
    for t in F.otype.s("text"):
        if F.text.v(t) == text_name:
            return L.d(t, otype="verse")
    raise KeyError(text_name)


def address(v):
    return (F.book.v(L.u(v, otype="book")[0]),
            F.chapter.v(L.u(v, otype="chapter")[0]), F.verse.v(v))


_dd = verses_of("cairo_genizah_targum_fragment_dd")
print(f"  cairo_genizah_targum_fragment_dd holds {len(_dd)} verses")
for _v in _dd[:4]:
    _b, _c, _vs = address(_v)
    print(f"    {_b} {_c}:{_vs:<3} {T.text(_v, fmt='text-trans-full')[:52]}")

In [ ]:
# The morphology under one surface word. This is what the dataset adds to a transcription: the
# surface word is split into the morphemes that compose it, each with its own lexeme and analysis.

_verse = T.nodeFromSection(("Genesis", 1, 1))
for _g in L.d(_verse, otype="word_group")[:4]:
    print(f"  {F.g_voc.v(_g):<14} {F.g_cons_utf8.v(_g)}")
    for _w in L.d(_g, otype="word"):
        print(f"      {F.voc.v(_w):<12} lex={F.lex.v(_w):<10} sp={F.sp.v(_w):<6} "
              f"gloss={(F.gloss.v(_w) or '')[:34]}")

## Comparing the witnesses

The question this corpus exists to answer is what the traditions do differently at the same verse.
A `crossref` edge joins every pair of verse nodes that share an address, so the comparison needs no
similarity measure and no threshold: it is given by the data.

The edge is labelled. `witness` joins two different manuscripts and is the parallel you usually
want. `view` joins Neofiti's two views of one codex, which is not a parallel and should not be
counted as a second witness.

In [ ]:
# Every witness to one verse, through the edge.

_node = T.nodeFromSection(("Genesis", 44, 18))
_b, _c, _vs = address(_node)
print(f"{_b} {_c}:{_vs}\n")
print(f"  {'[' + 'base' + ']':<10} {F.text.v(L.u(_node, otype='text')[0]):<34} "
      f"{len(L.d(_node, otype='word')):>4} words")
for _other, _relation in sorted(E.crossref.f(_node) or (),
                                key=lambda x: F.text.v(L.u(x[0], otype="text")[0])):
    _name = F.text.v(L.u(_other, otype="text")[0])
    print(f"  {_relation:<10} {_name:<34} {len(L.d(_other, otype='word')):>4} words")

print()
print("and the opening of each, to see the traditions diverge:")
for _other, _relation in list(E.crossref.f(_node) or ())[:4]:
    _name = F.text.v(L.u(_other, otype="text")[0])
    print(f"  {_name:<34} {T.text(_other, fmt='text-trans-full')[:46]}")

## Searching

A search template is an indented description of a structural pattern. Each line names a node type,
optionally with feature conditions, and indentation means containment. The result is a tuple of
nodes per match, in the order the lines appear.

In [ ]:
# Three searches, from the simplest upward.

from collections import Counter, defaultdict

print("1  Genesis 1 across every witness that attests it")
_rows = defaultdict(list)
for _t, _b, _c, _v in S.search("""
text
  book book=Genesis
    chapter chapter=1
      verse
"""):
    _rows[F.text.v(_t)].append(_v)

print(f"     {'witness':<24} {'verses':>6} {'range':>8} {'words':>7}")
for _name in sorted(_rows, key=lambda n: (-len(_rows[n]), n)):
    _nums = sorted(F.verse.v(_v) for _v in _rows[_name] if F.verse.v(_v) is not None)
    _words = sum(len(L.d(_v, otype="word")) for _v in _rows[_name])
    print(f"     {_name:<24} {len(_nums):>6} "
          f"{f'{_nums[0]}-{_nums[-1]}':>8} {_words:>7,}")
print(f"     {len(_rows)} of {len(list(F.otype.s('text')))} witnesses reach Genesis 1, and no "
      f"Genizah fragment is among them:")
print("     the fragments survive in patches, which is what makes their coverage worth checking")
print("     before a negative claim rather than after one.")

print()
print("   the same verse in each tradition, which is the comparison the corpus is for:")
for _name in sorted(_rows):
    _first = [_v for _v in _rows[_name] if F.verse.v(_v) == 1]
    if _first:
        print(f"     {_name:<24} {T.text(_first[0], fmt='text-trans-full')[:48]}")

print()
print("2  how often each part of speech occurs")
for _sp, _n in F.sp.freqList()[:6]:
    print(f"     {_sp:<8} {_n:>8,}")

print()
print("3  a verb lexeme, counted by witness")
_by_witness = Counter()
for _verse, _word in S.search("""
verse
  word sp=verb lex=>MR[
"""):
    _by_witness[F.text.v(L.u(_verse, otype="text")[0])] += 1
for _name, _n in _by_witness.most_common(6):
    print(f"     {_name:<34} {_n:>6,}")

## What not to trust, and how to exclude it

Two things to read before quoting a figure from this corpus.

`docs/transcription.md` lists every feature with its coverage measured at the feature's own tier.
Several features are filled for only a small minority of their tier, and the figure is stated for
each one, so a query can be designed around the coverage rather than discovering it afterwards.

The baseline suite under `programs/tests` measures whether each tier of the edition composes the
tier above it. Run it with `pytest programs/tests -q`, or with
`python programs/tests/baseline_report.py` for every table in order followed by the gate. Where a
tier does not compose the one above it, the suite writes `data/exclusions/`, which names each unit
rather than merely counting it: a manifest, a per-unit worklist and a slot-level filter. Units are
keyed on their own identifiers rather than on node numbers, because node numbers are reassigned
every time the dataset is compiled. The register is generated rather than shipped, so it cannot
describe a build other than the one in your own clone.


## Where to go next

| to | read |
|---|---|
| know what a feature means | `docs/transcription.md`, or the descriptions in `config/xml.yml` |
| know what the corpus is, and what it is not | `docs/about.md` |
| see every figure measured and gated | `programs/tests`, run with `pytest programs/tests -q` |
| read the relational form | the archive under `sql/`, one loadable file per publication view |
| rebuild the dataset | `src/main/pipeline/tf_pipeline.py`, then `programs/xml_to_tf.py` |

The loading cell above is the whole of what is needed to begin. It reads the version from the
application configuration rather than from a literal, so a notebook of your own cannot end up
describing a different build from the one it has loaded.
